# Data Cleaning — Dataset Tingkat Kemiskinan Indonesia (Dataset Baru)

Dataset baru: `Data_Tingkat_Kemiskinan.xlsx`  
Format: `.xlsx` (bukan CSV seperti sebelumnya)

**Penyesuaian dari dataset lama ke dataset baru:**

| Kolom Lama | Kolom Baru |
|---|---|
| `Kab/Kota` | `Kabupaten_Kota` |
| `Provinsi` | *(tidak ada — dihapus dari referensi)* |
| `Persentase Penduduk Miskin (P0)...` | `Tingkat_Penduduk_Miskin` |
| `Rata-rata Lama Sekolah Penduduk 15+...` | `Rata-rata Lama Sekolah` |
| `Indeks Pembangunan Manusia` | `Indeks Pembangunan Gender` |
| `Umur Harapan Hidup (Tahun)` | `Usia Harapan Hidup` |
| `Pengeluaran per Kapita Disesuaikan...` | `PengeluaranPerKapita` |
| `PDRB atas Dasar Harga Konstan...` | `Produk Domestik Regional Bruto` |
| `Klasifikasi Kemiskinan` | *(tidak ada — dibuat dari binning kuartil)* |
| `Persentase sanitasi`, `air minum`, `Pengangguran`, `TPAK` | *(tidak ada di dataset baru — dihapus)* |

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize

## 2. Load Dataset Baru
Dataset baru berformat `.xlsx`, jadi pakai `pd.read_excel` bukan `pd.read_csv`.

In [ ]:
# Upload file Data_Tingkat_Kemiskinan.xlsx ke Colab terlebih dahulu
df = pd.read_excel('/content/Data_Tingkat_Kemiskinan.xlsx')

print('Shape data:', df.shape)
df.head()

## 3. Memahami Data

In [ ]:
# seeing the data
df.info()

In [ ]:
# untuk memastikan kira-kira kolom mana yang perlu diperhatikan
df.nunique()

Dataset baru sudah bertipe numerik dengan benar (float64/int64), tidak perlu konversi tipe seperti dataset lama yang bertipe object.

In [ ]:
# cek sebaran data kolom utama
df['Tingkat_Penduduk_Miskin'].unique()

In [ ]:
df['PengeluaranPerKapita'].unique()

In [ ]:
df['Usia Harapan Hidup'].unique()

Semua kolom numerik sudah bertipe yang benar, tidak perlu konversi `str.replace(',', '.')` seperti dataset lama.

## 4. Membuat Kolom `Klasifikasi Kemiskinan`

Dataset baru tidak memiliki kolom `Klasifikasi Kemiskinan` seperti dataset lama.
Kolom ini dibuat menggunakan **binning berbasis kuartil** dari `Tingkat_Penduduk_Miskin`:

- **1** = Tidak Miskin (Q1 terbawah, % miskin paling rendah)
- **2** = Cukup Miskin
- **3** = Miskin
- **4** = Sangat Miskin (Q4 teratas, % miskin paling tinggi)

In [ ]:
# Hitung kuartil sebagai batas bin
q1 = df['Tingkat_Penduduk_Miskin'].quantile(0.25)
q2 = df['Tingkat_Penduduk_Miskin'].quantile(0.50)
q3 = df['Tingkat_Penduduk_Miskin'].quantile(0.75)

print(f'Q1 (batas Tidak Miskin / Cukup Miskin) : {q1}')
print(f'Q2 (batas Cukup Miskin / Miskin)        : {q2}')
print(f'Q3 (batas Miskin / Sangat Miskin)        : {q3}')

In [ ]:
# Buat kolom Klasifikasi Kemiskinan
bins = [0, q1, q2, q3, df['Tingkat_Penduduk_Miskin'].max() + 1]
labels_cat = [1, 2, 3, 4]

df['Klasifikasi Kemiskinan'] = pd.cut(
    df['Tingkat_Penduduk_Miskin'],
    bins=bins,
    labels=labels_cat
)

# cek hasil
df['Klasifikasi Kemiskinan'].value_counts().sort_index()

In [ ]:
# drop baris yang Klasifikasi Kemiskinannya NaN (kalau ada edge case)
df.dropna(subset=['Klasifikasi Kemiskinan'], inplace=True)
df['Klasifikasi Kemiskinan'] = df['Klasifikasi Kemiskinan'].astype(int)

# cek kembali
df.info()

In [ ]:
# save checkpoint pertama
df.to_csv('klasifikasi kemiskinan.csv', index=False)

## 5. Missing Value Handling

In [ ]:
# missing value see
df.isnull().sum()

In [ ]:
# hapus seluruh data yang missing value pada semua kolom
df = df.dropna()

In [ ]:
# checking hasil
df.isnull().sum()

In [ ]:
# checking info datanya
df.info()

In [ ]:
df.to_csv('klasifikasi kemiskinan_missing clean.csv', index=False)

okee data sudah bersih dari missing value

## 6. Checking Duplicated Data

In [ ]:
# duplicate check
df.duplicated().sum()

termasuk aman ya karna ngga ada kolom id (terhitung tanpa duplicated)

## 7. Outliers Checking

In [ ]:
# check outliers data dan rotate gambarnya supaya bisa dilihat dengan jelas
# dataset baru punya 10 kolom, layout (2,5)
df.select_dtypes(include='number').plot(
    kind='box', subplots=True, layout=(2, 5),
    sharex=False, sharey=False, figsize=(20, 10), color='k'
)
plt.tight_layout()
plt.show()

In [ ]:
# Menampilkan data berdasarkan target untuk Tingkat_Penduduk_Miskin
sns.boxplot(x='Klasifikasi Kemiskinan', y='Tingkat_Penduduk_Miskin', data=df)
plt.show()
# drop outliersnya

In [ ]:
# menampilkan juga data berikutnya
sns.boxplot(x='Klasifikasi Kemiskinan', y='Rata-rata Lama Sekolah', data=df)
plt.show()
# data benar: orang rata-rata sekolah lebih lama tergolong ngga miskin (masuk akal)

In [ ]:
sns.boxplot(x='Klasifikasi Kemiskinan', y='PengeluaranPerKapita', data=df)
plt.show()
# bagus insightnya

In [ ]:
sns.boxplot(x='Klasifikasi Kemiskinan', y='Indeks Pembangunan Gender', data=df)
plt.show()
# bagus insightnya

In [ ]:
sns.boxplot(x='Klasifikasi Kemiskinan', y='Usia Harapan Hidup', data=df)
plt.show()
# insight bagus

In [ ]:
sns.boxplot(x='Klasifikasi Kemiskinan', y='Harapan Lama Sekolah', data=df)
plt.show()
# drop outliers

In [ ]:
sns.boxplot(x='Klasifikasi Kemiskinan', y='Produk Domestik Regional Bruto', data=df)
plt.show()
# di kapping winsorize (distribusi sangat skewed)

In [ ]:
sns.boxplot(x='Klasifikasi Kemiskinan', y='Indeks Kemahalan Konstruksi', data=df)
plt.show()
# drop

In [ ]:
sns.boxplot(x='Klasifikasi Kemiskinan', y='PengeluaranPerkapita_Rokok', data=df)
plt.show()
# cek

In [ ]:
# eksekusi outliers
# Mencari tahu data yang outliers dengan mendefinisikan fungsi
def outliers(data_out, drop=False):
    outliers_indices_list = []
    for each_feature in data_out.columns:
        feature_data = data_out[each_feature]
        Q1 = np.percentile(feature_data, 25.)
        Q3 = np.percentile(feature_data, 75.)
        IQR = Q3 - Q1
        outlier_step = IQR * 1.5
        outliers_idx = feature_data[
            ~((feature_data >= Q1 - outlier_step) & (feature_data <= Q3 + outlier_step))
        ].index.tolist()
        if not drop:
            print('For the feature {}, Num of Outliers is {}'.format(each_feature, len(outliers_idx)))
        if drop:
            df.drop(outliers_idx, inplace=True, errors='ignore')
            print('Outliers from {} feature removed'.format(each_feature))
    return outliers_indices_list

In [ ]:
# hapus beberapa fitur outliers yang sudah di seleksi tadi
# (kolom yang boxplot-nya tadi terlihat ada outlier signifikan)
outliers(df[[
    'Tingkat_Penduduk_Miskin',
    'Harapan Lama Sekolah',
    'Indeks Kemahalan Konstruksi',
    'PengeluaranPerkapita_Rokok'
]])

In [ ]:
# drop outliers
outliers(df[[
    'Tingkat_Penduduk_Miskin',
    'Harapan Lama Sekolah',
    'Indeks Kemahalan Konstruksi',
    'PengeluaranPerkapita_Rokok'
]], drop=True)

In [ ]:
# winsorize untuk Produk Domestik Regional Bruto (distribusinya sangat skewed)
col = 'Produk Domestik Regional Bruto'
df[col] = winsorize(df[col], limits=[0.01, 0.01])

In [ ]:
# cek ulang hasilnya
df.select_dtypes(include='number').plot(
    kind='box', subplots=True, layout=(2, 5),
    sharex=False, sharey=False, figsize=(20, 10), color='k'
)
plt.tight_layout()
plt.show()

## 8. Korelasi Antar Variabel

In [ ]:
# melihat korelasi antar variable untuk mencari feature yang penting
plt.figure(figsize=(12, 10))
# Exclude kolom non-numerik
cor = df.drop(columns=['Kabupaten_Kota'], errors='ignore').corr(numeric_only=True)
sns.heatmap(cor, annot=True, linewidth=.5, cmap='magma')
plt.title('Korelasi Antar Variable', fontsize=20)
plt.tight_layout()
plt.show()

## 9. Pilih Fitur & Scaling

In [ ]:
# patenkan fitur-fitur yang dipakai
# (disesuaikan dengan kolom dataset baru)
features = [
    'Tingkat_Penduduk_Miskin',
    'PengeluaranPerKapita',
    'Usia Harapan Hidup'
]

X = df[features]
X.describe()

In [ ]:
# scaling data
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
# lihat tampilan data after scaling
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
X_scaled_df.describe()

## 10. Simpan Output Final

In [ ]:
df.to_csv('klasifikasi kemiskinan_missing clean.csv', index=False)
print('Data cleaning selesai!')
print(f'Total baris: {len(df)}')
df.head()